# Marketing Strategy Analysis - US Superstore



---
## 1. Data Loading and Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 14

In [ ]:
# Load dataset - supports both .xls and local .csv copies
url = 'https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%205%20-%20Data%20Processing/W5D5%20-%20Mini-project%20-%20bis/US%20Superstore%20data.xls'

try:
    df = pd.read_excel(url)
    print('Loaded from remote URL.')
except Exception:
    df = pd.read_csv('superstore_dataset.csv', encoding='latin-1')
    print('Loaded from local file.')

print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Schema and missing values
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Cleaning
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicates after: ', df.duplicated().sum())

# Date conversion
for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

# Postal Code null fill
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0).astype(int)

# Feature engineering
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year']    = df['Order Date'].dt.year
df['Order Month']   = df['Order Date'].dt.month

print('\nSample after preprocessing:')
df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head()

In [ ]:
# ── Fonction reutilisable : graphique a barres horizontales ─────────────────
# RETOUR #1 : les graphiques Top 20 Etats (ventes), Top 20 Villes (ventes)
# et Top 20 Clients (ventes) partagent la meme structure de trace. On centralise
# la logique dans plot_barh() pour eviter la duplication et faciliter la maintenance.

def plot_barh(series_values, series_index, title, xlabel,
              color='#2980b9', highlight_neg=False,
              fmt_annotation='${:,.0f}', fmt_axis='${:,.0f}K',
              axis_divisor=1e3, figsize=(14, 8), ax=None):
    """
    Trace un graphique a barres horizontales standard.

    Parametres
    ----------
    series_values   : array-like  valeurs a tracer
    series_index    : array-like  labels (noms des barres)
    title           : str         titre du graphique
    xlabel          : str         label axe X
    color           : str         couleur des barres (defaut bleu)
    highlight_neg   : bool        si True, barres negatives en rouge
    fmt_annotation  : str         format des etiquettes de valeur
    fmt_axis        : str         format du formateur de l'axe X
    axis_divisor    : float       diviseur pour l'axe X (ex. 1e3 pour K)
    figsize         : tuple       taille de la figure
    ax              : plt.Axes    axe existant (cree si None)
    """
    return_fig = ax is None
    if return_fig:
        fig, ax = plt.subplots(figsize=figsize)

    if highlight_neg:
        colors = ['#c0392b' if v < 0 else color for v in series_values[::-1]]
    else:
        colors = color

    bars = ax.barh(series_index[::-1], series_values[::-1], color=colors)

    x_max = max(abs(v) for v in series_values)
    for bar, val in zip(bars, series_values[::-1]):
        offset = x_max * 0.005
        ax.text(val + offset,
                bar.get_y() + bar.get_height() / 2,
                fmt_annotation.format(val),
                va='center', fontsize=8.5)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: fmt_axis.format(x / axis_divisor))
    )
    ax.axvline(x=0, color='black', linewidth=0.6, alpha=0.5)

    if return_fig:
        plt.tight_layout()
        plt.show()


print('Fonction plot_barh() definie et prete a l\'emploi.')


---
## 2. Zone Analysis - State Performance

### 2.1 Which states have the most sales?

In [ ]:
state_summary = (
    df.groupby('State')[['Sales', 'Profit']]
    .sum()
    .sort_values('Sales', ascending=False)
)
top20_states = state_summary.head(20)

# Reutilisation de plot_barh (meme logique que Top 20 Villes et Top 20 Clients)
plot_barh(
    series_values = top20_states['Sales'].values,
    series_index  = top20_states.index,
    title         = 'Top 20 States by Total Sales',
    xlabel        = 'Total Sales (USD)',
    color         = '#2980b9',
)

print('Top 5 states by sales:')
print(top20_states['Sales'].head().to_string())


### 2.2 New York vs California - Sales and Profit Comparison

In [ ]:
ny_ca = (
    df[df['State'].isin(['New York', 'California'])]
    .groupby('State')[['Sales', 'Profit']]
    .sum()
)
ny_ca['Profit Margin (%)'] = (ny_ca['Profit'] / ny_ca['Sales'] * 100).round(2)

print('New York vs California:')
print(ny_ca.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
metrics = ['Sales', 'Profit']
palette = ['#2980b9', '#27ae60']

for ax, metric, color in zip(axes, metrics, palette):
    bars = ax.bar(ny_ca.index, ny_ca[metric], color=[color, color], width=0.45,
                  edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, ny_ca[metric]):
        label = f'${val:,.0f}'
        ypos  = val + ny_ca[metric].max() * 0.01 if val >= 0 else val - ny_ca[metric].max() * 0.04
        ax.text(bar.get_x() + bar.get_width() / 2, ypos, label,
                ha='center', fontsize=11, fontweight='bold')
    ax.set_title(f'{metric} - New York vs California')
    ax.set_ylabel(f'{metric} (USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.grid(axis='y', alpha=0.4)

fig.suptitle('New York vs California: Sales and Profit Comparison', fontsize=15, fontweight='bold')
fig.tight_layout()
plt.show()

diff_sales  = ny_ca.loc['New York', 'Sales']  - ny_ca.loc['California', 'Sales']
diff_profit = ny_ca.loc['New York', 'Profit'] - ny_ca.loc['California', 'Profit']
print(f'\nSales difference (NY - CA):  ${diff_sales:,.0f}')
print(f'Profit difference (NY - CA): ${diff_profit:,.0f}')
print(f'NY profit margin:   {ny_ca.loc["New York", "Profit Margin (%)"]:.1f}%')
print(f'CA profit margin:   {ny_ca.loc["California", "Profit Margin (%)"]:.1f}%')

### 2.3 Outstanding Customer in New York

In [ ]:
ny_customers = (
    df[df['State'] == 'New York']
    .groupby('Customer Name')[['Sales', 'Profit']]
    .sum()
    .sort_values('Sales', ascending=False)
)
ny_customers['Profit Margin (%)'] = (ny_customers['Profit'] / ny_customers['Sales'] * 100).round(2)

top10_ny = ny_customers.head(10)

fig, ax = plt.subplots(figsize=(13, 6))
bar_colors = ['#c0392b' if v < 0 else '#2980b9' for v in top10_ny['Sales']]
bars = ax.bar(top10_ny.index, top10_ny['Sales'], color=bar_colors, edgecolor='white')

for bar, val in zip(bars, top10_ny['Sales']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + top10_ny['Sales'].max() * 0.01,
            f'${val:,.0f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_title('Top 10 Customers in New York by Sales')
ax.set_xlabel('Customer Name')
ax.set_ylabel('Total Sales (USD)')
ax.tick_params(axis='x', rotation=35)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
fig.tight_layout()
plt.show()

top_customer = ny_customers.index[0]
top_sales    = ny_customers['Sales'].iloc[0]
top_profit   = ny_customers['Profit'].iloc[0]
print(f'Outstanding customer in New York: {top_customer}')
print(f'  Total Sales:  ${top_sales:,.0f}')
print(f'  Total Profit: ${top_profit:,.0f}')
print(f'  Profit Margin: {ny_customers["Profit Margin (%)"].iloc[0]:.1f}%')

### 2.4 State-Level Profitability Differences

In [ ]:
state_profit = (
    df.groupby('State')[['Sales', 'Profit']]
    .sum()
    .assign(**{'Profit Margin (%)': lambda x: (x['Profit'] / x['Sales'] * 100)})
    .sort_values('Profit', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(17, 9))

# Left: profit by state (all states)
ax = axes[0]
colors_profit = ['#27ae60' if v >= 0 else '#c0392b' for v in state_profit['Profit']]
ax.barh(state_profit.index[::-1], state_profit['Profit'][::-1], color=colors_profit[::-1])
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Total Profit by State')
ax.set_xlabel('Total Profit (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.tick_params(axis='y', labelsize=7)

# Right: profit margin by state
ax2 = axes[1]
colors_margin = ['#27ae60' if v >= 0 else '#c0392b' for v in state_profit['Profit Margin (%)']]
ax2.barh(state_profit.index[::-1], state_profit['Profit Margin (%)'][::-1], color=colors_margin[::-1])
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.set_title('Profit Margin (%) by State')
ax2.set_xlabel('Profit Margin (%)')
ax2.tick_params(axis='y', labelsize=7)

fig.suptitle('State-Level Profitability Analysis', fontsize=15, fontweight='bold')
fig.tight_layout()
plt.show()

losing_states = state_profit[state_profit['Profit'] < 0]
print(f'States with negative profit: {len(losing_states)}')
print(losing_states[['Sales', 'Profit', 'Profit Margin (%)']].to_string())

---
## 3. Pareto Principle - Customers and Profit



In [ ]:
customer_profit = (
    df.groupby('Customer Name')['Profit']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
customer_profit.columns = ['Customer Name', 'Total Profit']

# FILTRE NECESSAIRE — clients a profit positif uniquement.
# Raison : inclure des clients a profit negatif ferait diminuer le cumul apres
# son pic, rendant la courbe non-monotone. Le seuil 80 % deviendrait inatteignable
# de facon coherente et l'interpretation Pareto serait invalide mathematiquement.
# On travaille donc uniquement sur la base des clients qui contribuent positivement
# au profit total.
positive_customers = customer_profit[customer_profit['Total Profit'] > 0].copy()
positive_customers['Cumulative Profit']  = positive_customers['Total Profit'].cumsum()
positive_customers['Cumulative Profit %'] = (
    positive_customers['Cumulative Profit'] / positive_customers['Total Profit'].sum() * 100
)
positive_customers['Customer Rank %'] = (
    np.arange(1, len(positive_customers) + 1) / len(positive_customers) * 100
)

# Find the 20% threshold
threshold_20 = positive_customers[positive_customers['Customer Rank %'] <= 20].iloc[-1]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(positive_customers['Customer Rank %'],
        positive_customers['Cumulative Profit %'],
        color='#2980b9', linewidth=2.5, label='Cumulative Profit %')

ax.axvline(x=20, color='#c0392b', linestyle='--', linewidth=1.5, label='20% of customers')
ax.axhline(y=80, color='#27ae60', linestyle='--', linewidth=1.5, label='80% of profit')
ax.scatter([threshold_20['Customer Rank %']], [threshold_20['Cumulative Profit %']],
           s=100, color='#c0392b', zorder=5)
ax.annotate(
    f"Top 20% -> {threshold_20['Cumulative Profit %']:.1f}% of profit",
    xy=(threshold_20['Customer Rank %'], threshold_20['Cumulative Profit %']),
    xytext=(25, threshold_20['Cumulative Profit %'] - 8),
    fontsize=10, color='#c0392b',
    arrowprops=dict(arrowstyle='->', color='#c0392b')
)

ax.set_title('Pareto Analysis - Cumulative Profit by Customer Rank')
ax.set_xlabel('Cumulative % of Customers (ranked by profit, descending)')
ax.set_ylabel('Cumulative % of Total Profit')
ax.legend()
ax.grid(True, alpha=0.4)
fig.tight_layout()
plt.show()

print('Pareto - Customers vs Profit:')
print(f'  Profitable customers in dataset: {len(positive_customers)}')
print(f'  Top 20% of customers ({int(len(positive_customers)*0.2)} customers) '
      f'generate {threshold_20["Cumulative Profit %"]:.1f}% of cumulative profit')
pareto_holds = threshold_20['Cumulative Profit %'] >= 75
print(f'  Pareto principle broadly applies: {pareto_holds}')

---
## 4. City-Level Analysis

### 4.1 Top 20 Cities by Sales

In [ ]:
city_summary = (
    df.groupby('City')[['Sales', 'Profit']]
    .sum()
    .assign(**{'Profit Margin (%)': lambda x: (x['Profit'] / x['Sales'] * 100)})
)

top20_cities_sales  = city_summary.sort_values('Sales', ascending=False).head(20)
top20_cities_profit = city_summary.sort_values('Profit', ascending=False).head(20)

# Reutilisation de plot_barh (meme logique que Top 20 Etats et Top 20 Clients)
plot_barh(
    series_values = top20_cities_sales['Sales'].values,
    series_index  = top20_cities_sales.index,
    title         = 'Top 20 Cities by Total Sales',
    xlabel        = 'Total Sales (USD)',
    color         = '#2980b9',
)


### 4.2 Top 20 Cities by Profit

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
colors_p = ['#c0392b' if v < 0 else '#27ae60' for v in top20_cities_profit['Profit']]
bars = ax.barh(top20_cities_profit.index[::-1], top20_cities_profit['Profit'][::-1],
               color=colors_p[::-1])

for bar, val in zip(bars, top20_cities_profit['Profit'][::-1]):
    ax.text(val + top20_cities_profit['Profit'].max() * 0.005,
            bar.get_y() + bar.get_height() / 2,
            f'${val:,.0f}', va='center', fontsize=8)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Top 20 Cities by Total Profit')
ax.set_xlabel('Total Profit (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
fig.tight_layout()
plt.show()

### 4.3 Profitability Differences Between Top 20 Cities (Sales vs Profit Margin)

In [ ]:
# Compare the union of both top-20 lists
top40_cities = pd.Index(
    list(top20_cities_sales.index) + list(top20_cities_profit.index)
).unique()
compare_df = city_summary.loc[top40_cities].sort_values('Sales', ascending=False)

fig, ax = plt.subplots(figsize=(15, 8))
x = np.arange(len(compare_df))
width = 0.45

bar1 = ax.bar(x - width/2, compare_df['Sales'],  width, label='Sales',  color='#2980b9', alpha=0.85)
bar2 = ax.bar(x + width/2, compare_df['Profit'], width, label='Profit',
              color=['#c0392b' if v < 0 else '#27ae60' for v in compare_df['Profit']], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(compare_df.index, rotation=55, ha='right', fontsize=8)
ax.set_ylabel('USD')
ax.set_title('Sales vs Profit - Top Cities Comparison')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.axhline(y=0, color='black', linewidth=0.7)
ax.legend()
fig.tight_layout()
plt.show()

# Highlight cities in top-20 sales but NOT top-20 profit
sales_only = set(top20_cities_sales.index) - set(top20_cities_profit.index)
profit_only = set(top20_cities_profit.index) - set(top20_cities_sales.index)
print('High sales but NOT in top-20 profit (margin risk):')
print(compare_df.loc[list(sales_only), ['Sales', 'Profit', 'Profit Margin (%)']].sort_values('Sales', ascending=False).to_string())
print('\nHigh profit but NOT in top-20 sales (hidden gems):')
print(compare_df.loc[list(profit_only), ['Sales', 'Profit', 'Profit Margin (%)']].sort_values('Profit', ascending=False).to_string())

---
## 5. Customer Analysis

### 5.1 Top 20 Customers by Sales

In [ ]:
customer_sales = (
    df.groupby('Customer Name')[['Sales', 'Profit']]
    .sum()
    .sort_values('Sales', ascending=False)
)
customer_sales['Profit Margin (%)'] = (
    customer_sales['Profit'] / customer_sales['Sales'] * 100
)
top20_customers = customer_sales.head(20)

# Reutilisation de plot_barh (meme logique que Top 20 Etats et Top 20 Villes)
plot_barh(
    series_values = top20_customers['Sales'].values,
    series_index  = top20_customers.index,
    title         = 'Top 20 Customers by Total Sales',
    xlabel        = 'Total Sales (USD)',
    color         = '#2980b9',
)

print('Top 10 customers summary:')
print(top20_customers.head(10).to_string())


### 5.2 Pareto Principle - Customers and Sales (Cumulative Curve)

In [ ]:
all_customers = (
    df.groupby('Customer Name')['Sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
all_customers.columns = ['Customer Name', 'Total Sales']
all_customers['Cumulative Sales']  = all_customers['Total Sales'].cumsum()
all_customers['Cumulative Sales %'] = (
    all_customers['Cumulative Sales'] / all_customers['Total Sales'].sum() * 100
)
all_customers['Customer Rank %'] = (
    np.arange(1, len(all_customers) + 1) / len(all_customers) * 100
)

threshold_20_sales = all_customers[all_customers['Customer Rank %'] <= 20].iloc[-1]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(all_customers['Customer Rank %'],
        all_customers['Cumulative Sales %'],
        color='#2980b9', linewidth=2.5, label='Cumulative Sales %')

ax.axvline(x=20, color='#c0392b', linestyle='--', linewidth=1.5, label='20% of customers')
ax.axhline(y=80, color='#27ae60', linestyle='--', linewidth=1.5, label='80% of sales')
ax.scatter([threshold_20_sales['Customer Rank %']],
           [threshold_20_sales['Cumulative Sales %']],
           s=100, color='#c0392b', zorder=5)
ax.annotate(
    f"Top 20% -> {threshold_20_sales['Cumulative Sales %']:.1f}% of sales",
    xy=(threshold_20_sales['Customer Rank %'], threshold_20_sales['Cumulative Sales %']),
    xytext=(25, threshold_20_sales['Cumulative Sales %'] - 8),
    fontsize=10, color='#c0392b',
    arrowprops=dict(arrowstyle='->', color='#c0392b')
)

ax.fill_between(all_customers['Customer Rank %'],
                all_customers['Cumulative Sales %'],
                alpha=0.08, color='#2980b9')

ax.set_title('Pareto Analysis - Cumulative Sales by Customer Rank')
ax.set_xlabel('Cumulative % of Customers (ranked by sales, descending)')
ax.set_ylabel('Cumulative % of Total Sales')
ax.legend()
ax.grid(True, alpha=0.4)
fig.tight_layout()
plt.show()

print('Pareto - Customers vs Sales:')
print(f'  Total customers: {len(all_customers)}')
print(f'  Top 20% ({int(len(all_customers)*0.2)} customers) generate '
      f'{threshold_20_sales["Cumulative Sales %"]:.1f}% of total sales')
pareto_sales = threshold_20_sales['Cumulative Sales %'] >= 75
print(f'  Pareto principle broadly applies: {pareto_sales}')

---
## 6. Strategic Recommendations

This section consolidates the findings from all analyses into an actionable marketing strategy.

In [ ]:
# Summary metrics for the final report
total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
overall_margin = total_profit / total_sales * 100

top3_states_sales  = state_summary.head(3)
top3_states_profit = state_summary.sort_values('Profit', ascending=False).head(3)
loss_states        = state_summary[state_summary['Profit'] < 0]

top3_cities_sales  = city_summary.sort_values('Sales', ascending=False).head(3)
top3_cities_profit = city_summary.sort_values('Profit', ascending=False).head(3)

print('=== MARKETING STRATEGY REPORT ===')
print()
print('OVERALL PERFORMANCE')
print(f'  Total Revenue:         ${total_sales:>12,.0f}')
print(f'  Total Profit:          ${total_profit:>12,.0f}')
print(f'  Overall Profit Margin: {overall_margin:>11.1f}%')
print()
print('PRIORITY STATES (top 3 by sales):')
for state, row in top3_states_sales.iterrows():
    margin = row['Profit'] / row['Sales'] * 100
    print(f'  {state:<20} Sales: ${row["Sales"]:>10,.0f}  Profit: ${row["Profit"]:>8,.0f}  Margin: {margin:.1f}%')
print()
print('STATES WITH LOSSES (require discount or cost review):')
for state, row in loss_states.iterrows():
    margin = row['Profit'] / row['Sales'] * 100
    print(f'  {state:<20} Sales: ${row["Sales"]:>10,.0f}  Profit: ${row["Profit"]:>8,.0f}  Margin: {margin:.1f}%')
print()
print('PRIORITY CITIES (top 3 by sales):')
for city, row in top3_cities_sales.iterrows():
    print(f'  {city:<20} Sales: ${row["Sales"]:>10,.0f}  Profit: ${row["Profit"]:>8,.0f}')
print()
print('NY vs CA KEY INSIGHT:')
ny = ny_ca.loc['New York']
ca = ny_ca.loc['California']
print(f'  New York:   Sales ${ny["Sales"]:,.0f}  Profit ${ny["Profit"]:,.0f}  Margin {ny["Profit Margin (%)"]:.1f}%')
print(f'  California: Sales ${ca["Sales"]:,.0f}  Profit ${ca["Profit"]:,.0f}  Margin {ca["Profit Margin (%)"]:.1f}%')
print()
print('PARETO SUMMARY:')
print(f'  Top 20% of customers generate {threshold_20["Cumulative Profit %"]:.1f}% of profit')
print(f'  Top 20% of customers generate {threshold_20_sales["Cumulative Sales %"]:.1f}% of sales')

### Written Recommendations

**1. Concentrate marketing investment in the top-3 states by sales.**  
As shown in the *Top 20 States by Total Sales* chart (Section 2.1), California, New York, and Texas
dominate revenue generation by a wide margin. These states demonstrate market readiness and strong
purchase intent. Loyalty campaigns, premium product placement, and volume discounts targeted at
existing high-value customers in these markets will yield the highest ROI.

**2. Investigate and address loss-making states before expanding there.**  
As highlighted in the *State-Level Profitability Analysis* chart (Section 2.4), states such as
Texas, Ohio, and Pennsylvania record significant sales volumes yet generate negative profit —
indicating excessive discounting or high shipping costs relative to order value. Before increasing
marketing spend in these markets, a cost audit and a discount cap (20% maximum) should be enforced.

**3. Prioritise New York for high-margin premium campaigns.**  
The *New York vs California* comparison (Section 2.2) shows that New York generates a stronger
profit margin despite lower raw sales volume than California. The outstanding customer identified
in the *Top 10 Customers in New York* chart (Section 2.3) should be enrolled in a VIP programme.
California, despite higher gross revenue, warrants a margin-improvement initiative before
scaling spend further.

**4. Activate the Pareto customer segment with dedicated account management.**  
The *Pareto Analysis — Cumulative Profit by Customer Rank* curve (Section 3) confirms that the
top 20% of profitable customers contribute a disproportionate share of total profit. Similarly,
the *Pareto Analysis — Cumulative Sales by Customer Rank* curve (Section 5.2) shows the same
concentration on the sales side. Assigning a dedicated account manager, offering early access
to new products, and providing personalised pricing will reduce churn risk in this critical segment.

**5. Treat high-sales / low-profit cities as turnaround targets, not growth targets.**  
The *Profitability Differences Between Top 20 Cities* analysis (Section 4.3) reveals cities
that rank in the top-20 by sales yet post negative or near-zero profit margins. These locations
should be treated as margin-improvement cases: review discount policies and product mix before
committing additional marketing budget.

**6. Use the top-20 cities by profit as expansion templates.**  
The *Top 20 Cities by Total Profit* chart (Section 4.2) identifies cities with strong returns
on a smaller sales base — ideal candidates for increased marketing investment because the
operational model is already profitable and incremental sales volume can scale existing margin.
New York City leads both the sales and profit rankings, confirming it as the single highest-priority
city for expansion initiatives.
